# LMNA

Cohort: `LMNA`, Phenopacket Store v0.1.27 (n = 259).

LMNA is the most densely structured cohort in the Phenopacket Store: 1152 term pairs
tested, 205 significant after FDR correction. It is a pleiotropic locus producing
near-disjoint disease axes from a single gene:

- cardiac / skeletal muscle: Emery-Dreifuss muscular dystrophy 2, dilated cardiomyopathy 1A
- metabolic: Dunnigan familial partial lipodystrophy (FPLD2)
- progeroid forms

Correlation results are reported as clusters rather than individual pairs. The
interpretable signal comes from the synergy analysis against disease subtype.

## 1. Load the cohort

In [1]:
import re
from pathlib import Path

import pandas as pd
from ppktstore.registry import configure_phenopacket_registry

from phenosign import (
    PhenotypeDatasetBuilder,
    HPOCorrelationAnalyzer,
    SynergyAnalyzer,
    has_disease,
)

GENE = "LMNA"
PPKT_RELEASE = "0.1.27"
OUTDIR = Path("../case_studies/LMNA")
OUTDIR.mkdir(parents=True, exist_ok=True)

MIN_INDIVIDUALS = 30
MIN_TARGET_SIZE = 20

In [2]:
registry = configure_phenopacket_registry()
with registry.open_phenopacket_store(release=PPKT_RELEASE) as ps:
    phenopackets = list(ps.iter_cohort_phenopackets(GENE))

print(f"Loaded {len(phenopackets)} phenopackets")

Loaded 259 phenopackets


In [3]:
# has_disease works on the phenopacket directly, no GPSEA cohort needed here
dataset = PhenotypeDatasetBuilder(phenopackets).build(build_gpsea_cohort=False)

## 2. Disease composition

In [4]:
diseases_df, sex_df, genes_df, _ = dataset.describe_conditions()
diseases_df.to_csv(OUTDIR / "disease_composition_LMNA.csv", index=False)
diseases_df

,label,n_individuals
disease_id,,
OMIM:151660,"Lipodystrophy, familial partial, type 2",127 (49.0%)
OMIM:115200,"Cardiomyopathy, dilated, 1A",61 (23.6%)
OMIM:181350,"Emery-Dreifuss muscular dystrophy 2, autosomal...",41 (15.8%)
OMIM:176670,Hutchinson-Gilford progeria,15 (5.8%)
OMIM:613205,LMNA-related congenital muscular dystrophy,15 (5.8%)


## 3. Pairwise correlation

In [5]:
analyzer = HPOCorrelationAnalyzer(dataset=dataset)
corr_results = analyzer.compute_correlation_matrix()

corr_table = corr_results.results_table
n_sig = int((corr_table["adj_p_value"] < 0.05).sum())
print(f"tested pairs: {len(corr_table)} | significant (adj p < 0.05): {n_sig}")
corr_table.head()

Calculating pairwise correlation: 100%|██████████| 1913/1913 [00:01<00:00, 1184.43it/s]


tested pairs: 1152 | significant (adj p < 0.05): 205


,HPO_A,HPO_A_label,HPO_B,HPO_B_label,correlation,p_value,adj_p_value,n(A:E/B:E),n(A:E/B:O),n(A:O/B:E),n(A:O/B:O),n_individuals,n_pmids,pmids
416,HP:0002460,Distal muscle weakness,HP:0003701,Proximal muscle weakness,1.000000,1.716896e-28,1.977864e-25,69,0,0,35,104,8,10580070;10662742;10939567;11344241;12920062;1...
822,HP:0003701,Proximal muscle weakness,HP:0034392,Joint contracture,1.000000,8.576144e-26,4.939859e-23,50,0,0,38,88,4,10580070;10662742;10939567;12920062
359,HP:0001771,Achilles tendon contracture,HP:0003701,Proximal muscle weakness,0.977032,4.373833e-24,1.679552e-21,50,1,0,37,88,4,10580070;10662742;10939567;12920062
386,HP:0001771,Achilles tendon contracture,HP:0034391,Elbow contracture,0.916972,6.097535e-23,1.756090e-20,61,2,2,37,102,4,10580070;10662742;10939567;12920062
538,HP:0003202,Skeletal muscle atrophy,HP:0003701,Proximal muscle weakness,1.000000,1.127216e-22,2.183418e-20,50,0,0,30,80,4,10580070;10662742;10939567;12920062


In [6]:
corr_results.save_correlation_results(
    output_file=str(OUTDIR / "correlation_results_LMNA.csv")
)

In [7]:
sig = corr_table.query("adj_p_value < 0.05")
sig = sig.reindex(sig["correlation"].abs().sort_values(ascending=False).index)
sig[
    [
        "HPO_A_label",
        "HPO_B_label",
        "correlation",
        "adj_p_value",
        "n_individuals",
        "n_pmids",
    ]
].head(25)

,HPO_A_label,HPO_B_label,correlation,adj_p_value,n_individuals,n_pmids
416,Distal muscle weakness,Proximal muscle weakness,1.000000,1.977864e-25,104,8
407,Distal muscle weakness,Skeletal muscle atrophy,1.000000,1.245332e-15,69,4
354,Achilles tendon contracture,Gowers sign,1.000000,4.351870e-04,53,4
693,Gowers sign,Elbow contracture,1.000000,4.351870e-04,53,4
686,Gowers sign,Arrhythmia,-1.000000,8.293259e-04,42,4
537,Skeletal muscle atrophy,Gowers sign,1.000000,5.791855e-03,52,4
1146,Abnormal left ventricular ejection fraction,Joint contracture,-1.000000,5.605436e-08,37,3
694,Gowers sign,Joint contracture,1.000000,4.351870e-04,53,4
1073,Lipodystrophy,Abnormal circulating creatine kinase activity,-1.000000,3.099203e-11,48,4
401,Poor head control,Distal muscle weakness,1.000000,3.435876e-12,80,7


In [8]:
print("n_pmids across significant pairs:")
print(sig["n_pmids"].describe())
print("\nsingle-publication pairs:", int((sig["n_pmids"] == 1).sum()))

n_pmids across significant pairs:
count    205.000000
mean       4.053659
std        1.482595
min        1.000000
25%        3.000000
50%        4.000000
75%        4.000000
max        8.000000
Name: n_pmids, dtype: float64

single-publication pairs: 3


In [9]:
# threshold raised, the full matrix is unreadable at 205 significant pairs
corr_results.plot_correlation_heatmap_with_significance(
    title_name="Cohort LMNA",
    corr_threshold=0.7,
    adj_pval_threshold=0.05,
)

## 4. Synergy per disease subtype

In [10]:
print("columns:", list(diseases_df.columns), "| index:", diseases_df.index.name)

comp = diseases_df.reset_index()

id_col = "disease_id" if "disease_id" in comp.columns else comp.columns[0]
label_col = "label" if "label" in comp.columns else comp.columns[1]
count_col = "n_individuals" if "n_individuals" in comp.columns else comp.columns[-1]


def parse_count(value):
    # counts come through formatted as '127 (49.0%)'
    if isinstance(value, (int, float)):
        return int(value)
    match = re.match(r"\s*(\d+)", str(value))
    if not match:
        raise ValueError(f"could not parse a count from {value!r}")
    return int(match.group(1))


targets = [
    (row[id_col], row[label_col], parse_count(row[count_col]))
    for _, row in comp.iterrows()
    if parse_count(row[count_col]) >= MIN_TARGET_SIZE
]

for omim_id, label, n in targets:
    print(f"{omim_id}  n={n:>4}  {label}")

columns: ['label', 'n_individuals'] | index: disease_id
OMIM:151660  n= 127  Lipodystrophy, familial partial, type 2
OMIM:115200  n=  61  Cardiomyopathy, dilated, 1A
OMIM:181350  n=  41  Emery-Dreifuss muscular dystrophy 2, autosomal dominant


In [11]:
synergy_analyzer = SynergyAnalyzer(
    dataset=dataset,
    min_individuals_for_synergy_calculation=MIN_INDIVIDUALS,
)

synergy_results = {}

for omim_id, label, n in targets:
    slug = omim_id.replace(":", "_")
    print(f"\n=== {omim_id} ({label}), n={n} ===")

    condition = dataset.get_condition(
        has_disease(omim_id),
        name=f"{label} vs other LMNA",
    )
    print(condition.value_counts(dropna=False).to_dict())

    try:
        res = synergy_analyzer.compute_synergy_matrix(condition=condition)
    except ValueError as exc:
        print(f"skipped: {exc}")
        continue

    synergy_results[omim_id] = res
    tbl = res.results_table

    # save_synergy_results() drops everything below synergy >= 0.01,
    # so keep the unfiltered table for the dissynergy side
    tbl.to_csv(OUTDIR / f"synergy_full_LMNA_{slug}.csv", index=False)
    res.save_synergy_results(
        output_file=str(OUTDIR / f"synergy_matrix_LMNA_{slug}.csv")
    )

    print(
        f"rows={len(tbl)} | synergy {tbl['synergy'].min():.4f} to {tbl['synergy'].max():.4f}"
        f" | positive={(tbl['synergy'] > 0).sum()} negative={(tbl['synergy'] < 0).sum()}"
    )


=== OMIM:151660 (Lipodystrophy, familial partial, type 2), n=127 ===
{0.0: 132, 1.0: 127}


Calculating pairwise synergy: 100%|██████████| 1623/1623 [00:05<00:00, 276.94it/s]


rows=101 | synergy -0.3844 to 0.1182 | positive=25 negative=76

=== OMIM:115200 (Cardiomyopathy, dilated, 1A), n=61 ===
{0.0: 198, 1.0: 61}


Calculating pairwise synergy: 100%|██████████| 1623/1623 [06:11<00:00,  4.37it/s]


rows=608 | synergy -0.9087 to 0.3619 | positive=91 negative=517

=== OMIM:181350 (Emery-Dreifuss muscular dystrophy 2, autosomal dominant), n=41 ===
{0.0: 218, 1.0: 41}


Calculating pairwise synergy: 100%|██████████| 1623/1623 [06:16<00:00,  4.32it/s]


rows=560 | synergy -0.9086 to 0.3594 | positive=68 negative=492


## 5. Synergy heatmaps

In [12]:
for omim_id, res in synergy_results.items():
    print(f"### {omim_id} ###")
    try:
        fig = res.plot_synergy_heatmap()
    except ValueError as exc:
        print(f"no heatmap: {exc}")
        continue
    if fig is not None:
        display(fig)

### OMIM:151660 ###


### OMIM:115200 ###


### OMIM:181350 ###


## 6. Top synergy and dissynergy per target

The heatmap masks values below `synergy_threshold` and sets `zmin=0`, so the
dissynergy side has to be read from the tables.

In [13]:
COLS = ["HPO_A_label", "HPO_B_label", "synergy", "adj_p_value", "n_individuals", "n_pmids"]

for omim_id, res in synergy_results.items():
    tbl = res.results_table
    print(f"\n########## {omim_id} ##########")
    print("--- top synergy ---")
    display(tbl.nlargest(10, "synergy")[COLS])
    print("--- top dissynergy ---")
    display(tbl.nsmallest(10, "synergy")[COLS])


########## OMIM:151660 ##########
--- top synergy ---


,HPO_A_label,HPO_B_label,synergy,adj_p_value,n_individuals,n_pmids
46,Achilles tendon contracture,Atrial fibrillation,0.118223,0.002244,79,5
36,Dilated cardiomyopathy,First degree atrioventricular block,0.064379,0.000918,151,4
74,Atrial fibrillation,Joint contracture,0.059202,0.036106,74,5
31,Dilated cardiomyopathy,Achilles tendon contracture,0.045835,0.005509,67,4
23,Cardiomyopathy,Achilles tendon contracture,0.040890,0.000918,43,4
14,Congestive heart failure,Dilated cardiomyopathy,0.039854,0.226796,51,4
45,Achilles tendon contracture,Scoliosis,0.037015,0.143510,46,2
18,Congestive heart failure,Generalized lipodystrophy,0.033188,0.139285,51,4
26,Cardiomyopathy,Generalized lipodystrophy,0.029166,0.144478,43,4
13,Congestive heart failure,Cardiomyopathy,0.026704,0.116306,39,4


--- top dissynergy ---


,HPO_A_label,HPO_B_label,synergy,adj_p_value,n_individuals,n_pmids
27,Cardiomyopathy,Lipodystrophy,-0.384383,0.000918,48,6
35,Dilated cardiomyopathy,Lipodystrophy,-0.328377,0.000918,165,6
40,Atrioventricular block,Lipodystrophy,-0.251805,0.000918,164,5
75,Progeroid facial appearance,Generalized lipodystrophy,-0.108346,0.002244,67,4
24,Cardiomyopathy,Atrial fibrillation,-0.091944,0.001554,43,4
96,Lipodystrophy,First degree atrioventricular block,-0.090706,0.000918,171,4
76,Progeroid facial appearance,Lipodystrophy,-0.088842,0.000918,70,5
97,Lipodystrophy,Second degree atrioventricular block,-0.064113,0.000918,176,5
42,Third degree atrioventricular block,Lipodystrophy,-0.054639,0.000918,176,5
21,Cardiomyopathy,Atrioventricular block,-0.047283,0.023507,34,5



########## OMIM:115200 ##########
--- top synergy ---


,HPO_A_label,HPO_B_label,synergy,adj_p_value,n_individuals,n_pmids
385,Proximal muscle weakness,Lipodystrophy,0.361936,0.000281,107,7
200,Distal muscle weakness,Lipodystrophy,0.358867,0.000281,94,7
60,Muscle weakness,Lipodystrophy,0.313017,0.000281,117,7
248,Elevated circulating creatine kinase activity,Lipodystrophy,0.261508,0.000281,66,3
211,Distal muscle weakness,Abnormal circulating creatine kinase activity,0.216876,0.003673,36,4
508,Proximal upper limb muscle weakness,Lipodystrophy,0.203470,0.000281,110,7
397,Proximal muscle weakness,Abnormal circulating creatine kinase activity,0.186912,0.001818,44,4
308,Gowers sign,Lipodystrophy,0.156683,0.000281,72,7
75,Muscle weakness,Abnormal circulating creatine kinase activity,0.146323,0.004639,47,4
199,Distal muscle weakness,Generalized lipodystrophy,0.118635,0.000281,77,5


... (2 outputs left out)


########## OMIM:181350 ##########
--- top synergy ---


,HPO_A_label,HPO_B_label,synergy,adj_p_value,n_individuals,n_pmids
502,Lipodystrophy,Abnormal left ventricular ejection fraction,0.359353,0.000259,52,4
558,Abnormal left ventricular ejection fraction,Abnormal circulating creatine kinase activity,0.201854,0.000259,36,3
537,Reduced left ventricular ejection fraction,Abnormal circulating creatine kinase activity,0.201854,0.000259,36,3
60,Muscle weakness,Abnormal left ventricular ejection fraction,0.180778,0.001307,51,4
498,Lipodystrophy,Reduced left ventricular ejection fraction,0.102228,0.000259,61,4
55,Muscle weakness,Reduced left ventricular ejection fraction,0.099779,0.008009,59,4
503,Lipodystrophy,Joint contracture,0.059042,0.000259,103,5
495,Generalized lipodystrophy,Joint contracture,0.059042,0.000259,103,5
310,Scapuloperoneal amyotrophy,Highly elevated creatine kinase,0.056046,0.041391,50,2
543,Moderately reduced left ventricular ejection f...,Abnormal circulating creatine kinase activity,0.055558,0.000259,37,3


--- top dissynergy ---


,HPO_A_label,HPO_B_label,synergy,adj_p_value,n_individuals,n_pmids
122,Achilles tendon contracture,Proximal muscle weakness,-0.908597,0.000259,88,4
355,Proximal muscle weakness,Elbow contracture,-0.850536,0.000259,88,4
113,Achilles tendon contracture,Distal muscle weakness,-0.826030,0.000259,75,4
167,Distal muscle weakness,Elbow contracture,-0.825985,0.000259,75,4
556,Abnormal left ventricular ejection fraction,Joint contracture,-0.820438,0.000259,37,3
174,Skeletal muscle atrophy,Elevated circulating creatine kinase activity,-0.757888,0.000259,45,2
182,Skeletal muscle atrophy,Proximal upper limb muscle weakness,-0.757060,0.000259,81,4
39,Muscle weakness,Elevated circulating creatine kinase activity,-0.747266,0.000259,65,3
146,Distal muscle weakness,Elevated circulating creatine kinase activity,-0.743696,0.000259,55,3
200,Elevated circulating creatine kinase activity,Proximal muscle weakness,-0.742334,0.000259,64,3


In [14]:
if synergy_results:
    combined = pd.concat(
        [res.results_table.assign(target=omim_id) for omim_id, res in synergy_results.items()],
        ignore_index=True,
    )
    combined.to_csv(OUTDIR / "synergy_all_targets_LMNA.csv", index=False)
    print(combined.groupby("target")["synergy"].describe())

             count      mean       std       min       25%       50%  \
target                                                                 
OMIM:115200  608.0 -0.082847  0.154792 -0.908682 -0.097499 -0.036850   
OMIM:151660  101.0 -0.017220  0.061933 -0.384383 -0.019124 -0.008317   
OMIM:181350  560.0 -0.104529  0.174405 -0.908597 -0.113220 -0.039221   

                  75%       max  
target                           
OMIM:115200 -0.009199  0.361936  
OMIM:151660 -0.000057  0.118223  
OMIM:181350 -0.012702  0.359353  


## 7. Variant-effect target

Adds a genotype axis alongside the disease-subtype axis. Needs
`build_gpsea_cohort=True`, which is considerably slower.

In [15]:
# from phenosign import has_variant_effect
#
# dataset_gpsea = PhenotypeDatasetBuilder(phenopackets).build(build_gpsea_cohort=True)
# condition_lof = dataset_gpsea.get_condition(
#     has_variant_effect("FRAMESHIFT_VARIANT"),
#     name="Truncating vs other LMNA",
# )
# res_lof = SynergyAnalyzer(
#     dataset=dataset_gpsea,
#     min_individuals_for_synergy_calculation=MIN_INDIVIDUALS,
# ).compute_synergy_matrix(condition=condition_lof)
# res_lof.plot_synergy_heatmap()
# res_lof.results_table.to_csv(OUTDIR / "synergy_full_LMNA_truncating.csv", index=False)